In [0]:
from pyspark.sql.functions import col, to_timestamp, from_json, element_at, map_values, explode, date_trunc
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, MapType, ArrayType

# ==========================================
# PART 1: Incremental Reads (Streams)
# ==========================================
bronze_power_stream = spark.readStream.table("default.bronze_power_data")
bronze_weather_stream = spark.readStream.table("default.bronze_weather_data")

# ==========================================
# PART 2: Schemas & Transformations
# ==========================================

# Define the nested array schema for the Power API
power_json_schema = StructType([
    StructField("data", ArrayType(
        StructType([
            StructField("period", StringType(), True),
            StructField("type", StringType(), True),
            StructField("value", StringType(), True)
        ])
    ), True)
])

# Define the expected JSON schema for weather
main_json_schema = StructType([
    StructField("temp", DoubleType(), True)
])

# Clean Power Data
silver_power_stream = (
    bronze_power_stream
    .filter(col("response").isNotNull())
    # Parse the JSON string
    .withColumn("parsed_response", from_json(col("response"), power_json_schema))
    .withColumn("reading", explode(col("parsed_response.data")))
    .filter(col("reading.type") == "D")
    # Extract and cast, defining the exact time format (yyyy-MM-dd'T'HH)
    .withColumn("timestamp", to_timestamp(col("reading.period"), "yyyy-MM-dd'T'HH"))
    .withColumn("demand_mw", col("reading.value").cast("double"))
    # Final cleanup
    .dropna(subset=["demand_mw", "timestamp"])
    .dropDuplicates(["timestamp"])
    .select("timestamp", "demand_mw")
)

# Clean Weather Data
silver_weather_stream = (
    bronze_weather_stream
    # Parse the raw JSON strings
    .withColumn("main_data", from_json(col("main"), main_json_schema))
    .withColumn("rain_data", from_json(col("rain"), MapType(StringType(), DoubleType())))
    
    # Cast the string to a long, then timestamp, then truncate to the hour
    .withColumn("raw_timestamp", col("dt").cast("long").cast("timestamp"))
    .withColumn("timestamp", date_trunc("hour", col("raw_timestamp")))
    
    # Extract temperatures and rain
    .withColumn("temperature", col("main_data.temp").cast("double"))
    .withColumn("precipitation", element_at(map_values(col("rain_data")), 1).cast("double"))
    
    # Clean up missing data
    .fillna({"precipitation": 0.0})
    .dropna(subset=["temperature", "timestamp"])
    
    # Deduplicate based on the newly aligned hourly timestamp
    .dropDuplicates(["timestamp"])
    
    # Keep only the final silver columns
    .select("timestamp", "temperature", "precipitation")
)

# ==========================================
# PART 3: Cost-Optimized Writes
# ==========================================

# Write Power incrementally (_v2 checkpoint to force a re-read)
power_write_query = (
    silver_power_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "s3://sarasota-power-data-bronze/checkpoints/silver_power_v4")
    .trigger(availableNow=True)
    .toTable("default.silver_power")
)

# Write Weather incrementally
weather_write_query = (
    silver_weather_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "s3://sarasota-power-data-bronze/checkpoints/silver_weather_v4")
    .trigger(availableNow=True)
    .toTable("default.silver_weather")
)

# Ensure the cluster doesn't shut down before the streams finish processing
power_write_query.awaitTermination()
weather_write_query.awaitTermination()